# nnU-Net을 사용한 모델 학습

In [ ]:
# 1단계: nnU-Net 설치
# GitHub에서 클론
!git clone https://github.com/MIC-DKFZ/nnUNet.git
%cd nnUNet
!pip install -e .

Cloning into 'nnUNet'...
remote: Enumerating objects: 14039, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 14039 (delta 14), reused 9 (delta 9), pack-reused 14013 (from 3)
Receiving objects: 100% (14039/14039), 8.63 MiB | 31.68 MiB/s, done.
Resolving deltas: 100% (10717/10717), done.
/content/nnUNet
Obtaining file:///content/nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metada

In [ ]:
import os
os.environ['nnUNet_raw'] = "/content/nnUNet_raw"
os.environ['nnUNet_preprocessed'] = "/content/nnUNet_preprocessed"
os.environ['nnUNet_results'] = "/content/nnUNet_results"

# 디렉토리 생성
!mkdir -p $nnUNet_raw $nnUNet_preprocessed $nnUNet_results

### 구글 드라이브에서 데이터셋 불러오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# tar 파일 위치 확인
!ls /content/drive/MyDrive/hippocampus/

Mounted at /content/drive
Task04_Hippocampus.tar


In [ ]:
# Colab 로컬에 압축 해제 (드라이브보다 빠름)
!tar -xf /content/drive/MyDrive/hippocampus/Task04_Hippocampus.tar -C /content/

In [ ]:
# 압축 해제된 데이터 구조 확인
!ls /content/Task04_Hippocampus/

dataset.json  imagesTr	imagesTs  labelsTr


In [ ]:
#nnU-Net 형식으로 데이터 복사
# Dataset004로 복사
!cp -r /content/Task04_Hippocampus $nnUNet_raw/Dataset004_Hippocampus

# 구조 확인
!ls $nnUNet_raw/Dataset004_Hippocampus/

dataset.json  imagesTr	imagesTs  labelsTr


In [ ]:
# 데이터 검증 및 전처리
# 데이터셋 무결성 확인 및 전처리 계획 수립
!nnUNetv2_plan_and_preprocess -d 004 --verify_dataset_integrity

Fingerprint extraction...
Dataset004_Hippocampus
Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_plan_and_preprocess", line 8, in <module>
    sys.exit(plan_and_preprocess_entry())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/nnUNet/nnunetv2/experiment_planning/plan_and_preprocess_entrypoints.py", line 180, in plan_and_preprocess_entry
    extract_fingerprints(args.d, args.fpe, args.npfp, args.verify_dataset_integrity, args.clean, args.verbose)
  File "/content/nnUNet/nnunetv2/experiment_planning/plan_and_preprocess_api.py", line 47, in extract_fingerprints
    extract_fingerprint_dataset(d, fingerprint_extractor_class, num_processes, check_dataset_integrity, clean,
  File "/content/nnUNet/nnunetv2/experiment_planning/plan_and_preprocess_api.py", line 30, in extract_fingerprint_dataset
    verify_dataset_integrity(join(nnUNet_raw, dataset_name), num_processes)
  File "/content/nnUNet/nnunetv2/experiment_planning/verify_dataset_integrity.py", line 135, in

In [ ]:
# 트레이닝 이미지 개수 확인
!ls /content/Task04_Hippocampus/imagesTr/*.nii.gz | wc -l

260


In [ ]:
import json

# Hippocampus 데이터셋용 dataset.json 생성
dataset_json = {
    "channel_names": {
        "0": "MRI"
    },
    "labels": {
        "background": 0,
        "Anterior": 1,
        "Posterior": 2
    },
    "numTraining": 260,  # imagesTr 폴더의 실제 파일 개수 확인 필요
    "file_ending": ".nii.gz"
}

# 저장
with open('/content/nnUNet_raw/Dataset004_Hippocampus/dataset.json', 'w') as f:
    json.dump(dataset_json, f, indent=2)

print("dataset.json 생성 완료!")

dataset.json 생성 완료!


In [ ]:
# 실제 파일명 확인
!ls /content/nnUNet_raw/Dataset004_Hippocampus/imagesTr/ | head -20

# Mac 메타데이터 파일 전부 삭제
!find /content/nnUNet_raw/Dataset004_Hippocampus -name "._*" -delete
!find /content/nnUNet_raw/Dataset004_Hippocampus -name ".DS_Store" -delete

# 실제 .nii.gz 파일 개수 다시 확인
!ls /content/nnUNet_raw/Dataset004_Hippocampus/imagesTr/*.nii.gz 2>/dev/null | wc -l

hippocampus_001.nii.gz
hippocampus_003.nii.gz
hippocampus_004.nii.gz
hippocampus_006.nii.gz
hippocampus_007.nii.gz
hippocampus_008.nii.gz
hippocampus_011.nii.gz
hippocampus_014.nii.gz
hippocampus_015.nii.gz
hippocampus_017.nii.gz
hippocampus_019.nii.gz
hippocampus_020.nii.gz
hippocampus_023.nii.gz
hippocampus_024.nii.gz
hippocampus_025.nii.gz
hippocampus_026.nii.gz
hippocampus_033.nii.gz
hippocampus_034.nii.gz
hippocampus_035.nii.gz
hippocampus_036.nii.gz
260


In [ ]:
import os
from pathlib import Path

# imagesTr 파일명 변경
images_dir = Path('/content/nnUNet_raw/Dataset004_Hippocampus/imagesTr')
for f in sorted(images_dir.glob('hippocampus_*.nii.gz')):
    new_name = f.name.replace('.nii.gz', '_0000.nii.gz')
    f.rename(images_dir / new_name)

# labelsTr 파일명 변경 (레이블은 채널 번호 없음)
labels_dir = Path('/content/nnUNet_raw/Dataset004_Hippocampus/labelsTr')
# labelsTr은 그대로 유지

# imagesTs도 있다면 변경
images_ts_dir = Path('/content/nnUNet_raw/Dataset004_Hippocampus/imagesTs')
if images_ts_dir.exists():
    for f in sorted(images_ts_dir.glob('hippocampus_*.nii.gz')):
        new_name = f.name.replace('.nii.gz', '_0000.nii.gz')
        f.rename(images_ts_dir / new_name)

print("파일명 변경 완료!")

# 확인
!ls /content/nnUNet_raw/Dataset004_Hippocampus/imagesTr/ | head -5

파일명 변경 완료!
hippocampus_001_0000.nii.gz
hippocampus_003_0000.nii.gz
hippocampus_004_0000.nii.gz
hippocampus_006_0000.nii.gz
hippocampus_007_0000.nii.gz


In [ ]:
!nnUNetv2_plan_and_preprocess -d 004 --verify_dataset_integrity

Fingerprint extraction...
Dataset004_Hippocampus
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
100% 260/260 [00:04<00:00, 52.71it/s] 
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [36. 50. 35.], 3d_lowres: [36, 50, 35]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 366, 'patch_size': (np.int

In [ ]:
# 3D fullres 모델 학습 (fold 0)
!nnUNetv2_train 004 3d_fullres 0 --npz


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-01-19 08:27:28.970343: Using torch.compile...
2026-01-19 08:27:32.232604: do_dummy_2d_data_aug: False
2026-01-19 08:27:32.233512: Creating new 5-fold cross-validation split...
2026-01-19 08:27:32.236098: Desired fold for training: 0
2026-01-19 08:27:32.236172: This split has 208 training an